# DSPy Prompting

In [1]:
import json
from typing import List, Literal
import dspy
from pydantic import BaseModel, Field

In [7]:
# =====================================================================
# DIVISION 0: DATA MINING PATH CONFIGURATION
# =====================================================================
DATASAUR_FILE_PATH = "/workspaces/fil-srl/data/fil-srl-wf/SRL_WF_part1_from1_to815-sentence-MGUxN2VjZmQ/DOCUMENT-tagteam_xform_online/DOCUMENT-tagteam_xform_online-WF_part1_from1_to815-sentence.json"

DATASAUR_FILE_PATH = "/workspaces/fil-srl/data/fil-srl-wf/SRL_WF_part1_from1_to815-sentence-MGUxN2VjZmQ/DOCUMENT-yuananot_gmail_com/DOCUMENT-yuananot_gmail_com-WF_part1_from1_to815-sentence.json"

DATASAUR_FILE_PATH = "/workspaces/fil-srl/data/sample.json"

with open(DATASAUR_FILE_PATH, "r", encoding="utf-8") as file:
  raw_datasaur_data = json.load(file)

payload_data = raw_datasaur_data["data"]

# Extract all unique validation labels directly from your codebook metadata
fil_srl_labels = [
  label["labelName"] for label in payload_data["labelSets"][0]["labelItems"]
]

# Isolate vocabulary tags to prevent the LLM from inventing outside labels
# Isolate vocabulary tags to prevent the LLM from inventing outside labels
VERB_TAXONOMY_TAGS = [
  l
  for l in fil_srl_labels
  if l.startswith("REL-VRB-")
  or l
  in [
    "REL-ADJ",
    "REL-NOM",
    "REL-EXT",  # Existential
    "REL-MOD",  # Modal
    "REL-PREP",  # Prepositional
  ]
]
argument_labels = [l for l in fil_srl_labels if l not in VERB_TAXONOMY_TAGS]
FilSrlRoles = Literal[tuple(argument_labels)]

In [8]:
# =====================================================================
# DIVISION 1: DATA STRUCTURE & PYDANTIC OUTPUT SCHEMAS
# =====================================================================
class SemanticArgument(BaseModel):
  """Schema representing an individual semantic role argument extraction."""

  role: FilSrlRoles = Field(
    description="The exact categorical semantic role from your codebook."
  )
  text: str = Field(
    description="The exact phrase copied character-for-character from the sentence."
  )


class TagalogSRL(dspy.Signature):
  """
  Suriin ang buong pangungusap at kunin (extract) ang mga semantic argument role na nakadepende LAMANG sa ibinigay na Target na Pandiwa/Predicate (Target Verb).

  PATAKARAN: Kopyahin ang teksto nang eksakto (verbatim slice) mula sa orihinal na pangungusap.
  """

  sentence: str = dspy.InputField(desc="The full Tagalog sentence text to be analyzed.")
  target_verb: str = dspy.InputField(
    desc="The specific predicate verb text phrase to extract arguments for."
  )

  extracted_arguments: List[SemanticArgument] = dspy.OutputField(
    desc="A structured list of all identified semantic roles and their corresponding full text spans."
  )


In [9]:
# =====================================================================
# DIVISION 2: ENVIRONMENT CONFIGURATION & LOCAL OLLAMA BACKEND
# =====================================================================
OLLAMA_MODEL_TAG = "ollama_chat/qwen3.6:35b"

lm = dspy.LM(
  OLLAMA_MODEL_TAG,
  api_base="http://localhost:11434",
  api_key="",
  temperature=0.0,
  cache=False,
)

dspy.settings.configure(lm=lm)
srl_predictor = dspy.ChainOfThought(TagalogSRL)

In [10]:
# =====================================================================
# DIVISION 3: MULTI-VERB AUTOMATIC TASK DISCOVERY (NO HARDCODING)
# =====================================================================
# We parse the file to group sentences and auto-detect every active predicate
evaluation_tasks = []

try:
  row_id = 0

  # 1. Map text data attributes
  row_zero_object = payload_data["rows"][row_id][0]
  sentence_text = row_zero_object["content"]
  row_tokens = row_zero_object["tokens"]

  # Map out all spans indexed cleanly by their unique ID
  all_spans_by_id = {}
  row_predicates = []  # Holds every predicate found in this row

  for span in payload_data.get("spanLabels", []):
    if span["textPosition"]["start"]["row"] == row_id:
      span_id = span["id"]
      all_spans_by_id[span_id] = span

      role_label = span["labelItem"]["labelName"]
      start_idx = span["textPosition"]["start"]["tokenIndex"]
      end_idx = span["textPosition"]["end"]["tokenIndex"]
      raw_token_string = " ".join(row_tokens[start_idx : end_idx + 1]).strip()

      # DYNAMIC PREDICATE DISCOVERY: If the tag is part of your predicate codebook, log it!
      if role_label in VERB_TAXONOMY_TAGS:
        row_predicates.append({
          "span_id": span_id,
          "verb_text": raw_token_string,
          "tag": role_label,
        })

  # 2. For EACH discovered verb, follow its explicit arrows to capture its exact arguments
  for pred in row_predicates:
    target_verb_span_id = pred["span_id"]

    # Scan arrow relationships targeting this specific verb node instance
    valid_argument_ids = set()
    for arrow in payload_data.get("arrowLabels", []):
      if str(arrow["destinationId"]) == str(target_verb_span_id):
        valid_argument_ids.add(arrow["originId"])

    # Gather the ground truth arguments associated with this specific arrow loop
    verb_ground_truth = []
    for arg_id in valid_argument_ids:
      span = all_spans_by_id.get(arg_id)
      if span:
        arg_start = span["textPosition"]["start"]["tokenIndex"]
        arg_end = span["textPosition"]["end"]["tokenIndex"]
        raw_extracted_argument_text = " ".join(row_tokens[arg_start : arg_end + 1])

        verb_ground_truth.append({
          "role": span["labelItem"]["labelName"],
          "text": raw_extracted_argument_text,
          "start_idx": arg_start,
        })

    # Sort argument nodes left-to-right chronologically
    verb_ground_truth = sorted(verb_ground_truth, key=lambda x: x["start_idx"])

    # Save this task mapping to be executed in our final evaluation runner
    evaluation_tasks.append({
      "sentence": sentence_text,
      "target_verb": pred["verb_text"],
      "verb_tag": pred["tag"],
      "ground_truth": verb_ground_truth,
    })

  print(
    f"✅ Dynamic Discovery Complete! Detected {len(evaluation_tasks)} unique relations/predicates in Row {row_id}."
  )
  for idx, task in enumerate(evaluation_tasks):
    print(
      f"   Task #{idx + 1}: Predicate='{task['target_verb']}' (Taxonomy Code: {task['verb_tag']}) -> Expects {len(task['ground_truth'])} attached arguments."
    )

except KeyError as e:
  raise KeyError(f"❌ Dynamic discovery structure mapping error: {str(e)}")


✅ Dynamic Discovery Complete! Detected 1 unique relations/predicates in Row 0.
   Task #1: Predicate='lamang sa pagiging mura' (Taxonomy Code: REL-ADJ) -> Expects 9 attached arguments.


In [13]:
# =====================================================================
# DIVISION 4: TWIN CHRONOLOGICAL TABLES EVALUATION MATRIX
# =====================================================================
print(
  f"🔄 Contacting local Ollama engine ({OLLAMA_MODEL_TAG}) via DSPy configuration framework..."
)


def normalize_text(text: str) -> str:
  """Removes all spaces and standardizes case to prevent tokenization spacing mismatches."""
  if not text:
    return ""
  return text.replace(" ", "").lower()


for task in evaluation_tasks:
  target_action = task["target_verb"]
  datasaur_ground_truth = task["ground_truth"]
  verb_tag = task["verb_tag"]

  print(f"\n🎯 EXECUTING TASK: Evaluating arguments for Predicate -> '{target_action}'")

  try:
    # Run structured evaluation task prediction pipeline loop
    prediction = srl_predictor(sentence=sentence_text, target_verb=target_action)
    llm_arguments = prediction.extracted_arguments

    def get_token_start_index(text_span: str) -> int:
      norm_span = normalize_text(text_span)
      norm_sentence = normalize_text(sentence_text)
      if norm_span and norm_span in norm_sentence:
        return norm_sentence.index(norm_span)
      return 999

    # Sort predicted outputs from left-to-right linearly
    ordered_llm_predictions = sorted(
      llm_arguments, key=lambda x: get_token_start_index(x.text)
    )

    # -----------------------------------------------------------------
    # TABLE 1: REFACTORED MULTI-SPAN GROUND TRUTH BASELINE
    # -----------------------------------------------------------------
    # INJECT THE PREDICATE INTO THE BASELINE FOR SEAMLESS DISPLAY
    chronological_baseline = list(datasaur_ground_truth)
    chronological_baseline.append({
      "role": f"⭐ {verb_tag} (TARGET)",
      "text": target_action,
      "is_predicate": True,
    })

    # Sort the newly combined list chronologically
    chronological_baseline = sorted(
      chronological_baseline, key=lambda x: get_token_start_index(x["text"])
    )

    print("\n" + "=" * 125)
    print(f"📋 TABLE 1: ARROW-FILTERED DATASAUR BASELINE (Chronological Sentence Flow)")
    print("=" * 125)
    print(
      f"{'Actual SRL Role':<25} | {'Ground Truth Text Span':<60} | Prediction Alignment Status"
    )
    print("-" * 125)

    for item in chronological_baseline:
      role = item["role"]
      actual_text = item["text"]

      # Special handling for the injected predicate row
      if item.get("is_predicate"):
        print(f"{role:<25} | {actual_text:<60} | 🔵 TARGET PREDICATE ANCHOR")
        continue

      # Normalized String Matching
      match_found = False
      matched_text = ""
      for pred in ordered_llm_predictions:
        if pred.role == role and normalize_text(pred.text) == normalize_text(
          actual_text
        ):
          match_found = True
          break
        elif pred.role == role:
          matched_text = pred.text

      if match_found:
        status = "✅ EXACT MATCH"
      elif matched_text:
        status = f"⚠️ SPAN SHIFT (LLM got: '{matched_text[:25]}...')"
      else:
        status = "❌ OMITTED / MISSED BY LLM"

      print(f"{role:<25} | {actual_text:<60} | {status}")
    print("=" * 125)

    # -----------------------------------------------------------------
    # TABLE 2: LLM OUTPUT PROFILE
    # -----------------------------------------------------------------
    print("\n" + "=" * 125)
    print(f"🤖 TABLE 2: LOCAL OLLAMA INTERPRETATION PROFILE ({OLLAMA_MODEL_TAG})")
    print("=" * 125)
    print(
      f"{'Predicted Role':<25} | {'LLM Extracted Verbatim Text Span':<60} | Ground Truth Validity"
    )
    print("-" * 125)

    for pred in ordered_llm_predictions:
      p_role = pred.role
      p_text = pred.text

      exact_match_valid = any(
        item["role"] == p_role
        and normalize_text(item["text"]) == normalize_text(p_text)
        for item in datasaur_ground_truth
      )
      role_exists_elsewhere = any(
        item["role"] == p_role for item in datasaur_ground_truth
      )

      if exact_match_valid:
        validity = "✅ VALID MATCH WITH GROUND TRUTH"
      elif role_exists_elsewhere:
        validity = "⚠️ VARIANT SPAN BOUNDARY (Expected different textual coverage slice)"
      else:
        validity = "❌ MISPLACED SPAN (Extra prediction not attached to this target)"

      print(f"{p_role:<25} | {p_text:<60} | {validity}")
    print("=" * 125)

  except Exception as e:
    print(f"\n❌ DSPy evaluation engine encountered a fatal break: {str(e)}")


🔄 Contacting local Ollama engine (ollama_chat/qwen3.6:35b) via DSPy configuration framework...

🎯 EXECUTING TASK: Evaluating arguments for Predicate -> 'lamang sa pagiging mura'

📋 TABLE 1: ARROW-FILTERED DATASAUR BASELINE (Chronological Sentence Flow)
Actual SRL Role           | Ground Truth Text Span                                       | Prediction Alignment Status
-----------------------------------------------------------------------------------------------------------------------------
ARGM-ADJ                  | Bilang isang paraan sa pagkontrol ng pag - aanak             | ❌ OMITTED / MISSED BY LLM
ARG1                      | ang mga kondom                                               | ❌ OMITTED / MISSED BY LLM
ARGM-ADJ                  | na para sa mga lalaki                                        | ❌ OMITTED / MISSED BY LLM
ARGM-LEX                  | ay                                                           | ❌ OMITTED / MISSED BY LLM
⭐ REL-ADJ (TARGET)        | lamang

# Zero-Shot Prompting

In [ ]:
import json
import requests

In [ ]:
DATASAUR_FILE_PATH = "/workspaces/fil-srl/data/fil-srl-wf/SRL_WF_part1_from1_to815-sentence-MGUxN2VjZmQ/DOCUMENT-tagteam_xform_online/DOCUMENT-tagteam_xform_online-WF_part1_from1_to815-sentence.json"

with open(DATASAUR_FILE_PATH, "r", encoding="utf-8") as file:
  raw_datasaur_data = json.load(file)

fil_srl_labels = list(
  set([
    label["labelName"]
    for label in raw_datasaur_data["data"]["labelSets"][0]["labelItems"]
  ])
)
VALID_ROLES = fil_srl_labels

In [ ]:
# =====================================================================
# REVISED DIVISION 1: RIGID TAXONOMY ZERO-SHOT SYSTEM PROMPT (FILIPINO)
# =====================================================================
VALID_ROLES = [
  "ARGM-PRD",
  "REL-ADJ",
  "ARG1",
  "ARGM-LEX",
  "ARGM-DIS",
  "ARGM-ADJ",
  "ARGM-ADV",
  "ARGM-LOC",
]

SYSTEM_PROMPT = f"""Ikaw ay isang dalubhasang linguistic parser na eksperto sa Filipino/Tagalog Semantic Role Labeling (SRL) gamit ang PropBank framework.
Ang iyong iisang tungkulin ay suriin ang pangungusap at kunin (extract) ang mga semantic argument role na nakadepende LAMANG sa ibinigay na Target na Pandiwa (Target Verb).

MGA MAHIGPIT NA PATAKARAN SA PAG-LABEL:
1. BAWAL MAG-IMBENTO NG ROLES. Ang bawat 'role' sa iyong output ay DAPAT piliin LAMANG mula sa eksaktong listahang ito: {VALID_ROLES}. Huwag na huwag gagamit ng mga tag tulad ng REL-NOM, REL-VRB-OBJ, ARG0, ARG2, ARGM-MOD, o ARGM-CAU. Kung hindi ito kasali sa listahan, BAWAL itong ibalik.
2. TUKUYIN ANG SAKLAW NG PARIRALA (SPAN BOUNDARIES):
   - Ang 'ARG1' para sa pandiwang relative clause na "nakukuha" ay ang pangunahing simuno na tinutukoy nito: "ang mga kondom". Huwag isasama ang buong mahabang listahan ng mga katangian sa ARG1.
   - Ang buong sugnay na naglalaman ng pandiwa, simula sa pang-angkop na 'na' hanggang sa dulo, ay dapat i-label bilang 'ARGM-PRD'. Halimbawa: "na nakukuha lamang sa pagtatalik (STDs)" -> ARGM-PRD.
3. Kopyahin ang teksto (text value) nang eksakto (character-for-character verbatim slice) mula sa pangungusap. Huwag magbabawas o magdaragdag ng mga bantas.

PAHAYAG NG ALINGAN (OUTPUT FORMAT):
Ibalik ang iyong sagot STRICTLY bilang isang raw JSON list ng mga object na naglalaman ng "role" at "text". Huwag magsusulat ng anumang pambungad na usapan, paliwanag, o markdown code block (bawal ang ```json o backticks)."""


In [ ]:
# =====================================================================
# DIVISION 2: OLLAMA DIRECT CONNECTION LAYER
# =====================================================================
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen3.6:35b"


def query_ollama_zero_shot(sentence: str, verb: str) -> list:
  user_prompt = f'Pangungusap: "{sentence}"\nTarget na Pandiwa: "{verb}"'

  payload = {
    "model": MODEL_NAME,
    "prompt": f"{SYSTEM_PROMPT}\n\n{user_prompt}",
    "stream": False,
    "options": {"temperature": 0.0},
  }

  print(
    f"🔄 Direct-pinging local Ollama engine ({MODEL_NAME}) using Filipino Prompt..."
  )
  response = requests.post(OLLAMA_URL, json=payload)
  response_json = response.json()
  raw_response_text = response_json.get("response", "").strip()

  # Strip unintentional markdown wrappers
  if raw_response_text.startswith("```"):
    raw_response_text = raw_response_text.split("\n", 1).rsplit("```", 1)[0].strip()
  if raw_response_text.startswith("json"):
    raw_response_text = raw_response_text[4:].strip()

  try:
    return json.loads(raw_response_text)
  except json.JSONDecodeError:
    print(
      f"⚠️ Ang model ay nagbalik ng maling format ng JSON text:\n{raw_response_text}"
    )
    return []


# =====================================================================
# DIVISION 3: DATASAUR FILE LOADER & GROUND TRUTH EXTRACTOR
# =====================================================================
DATASAUR_FILE_PATH = "/workspaces/fil-srl/data/fil-srl-wf/SRL_WF_part1_from1_to815-sentence-MGUxN2VjZmQ/DOCUMENT-tagteam_xform_online/DOCUMENT-tagteam_xform_online-WF_part1_from1_to815-sentence.json"

try:
  with open(DATASAUR_FILE_PATH, "r", encoding="utf-8") as file:
    raw_datasaur_data = json.load(file)

  payload_data = raw_datasaur_data["data"]

  # FIX: Corrected double list-nesting index pointer to pull Row 0 details cleanly
  row_zero_object = payload_data["rows"][0][0]
  sentence_text = row_zero_object["content"]
  row_tokens = row_zero_object["tokens"]

  target_action = "nakukuha"
  datasaur_ground_truth = []

  for span in payload_data.get("spanLabels", []):
    start_pos = span["textPosition"]["start"]
    end_pos = span["textPosition"]["end"]

    if start_pos["row"] == 0:
      role_label = span["labelItem"]["labelName"]
      if role_label in ["REL-VRB-REC", "V", "PREDICATE"]:
        continue

      start_token_idx = start_pos["tokenIndex"]
      end_token_idx = end_pos["tokenIndex"]

      token_slice = row_tokens[start_token_idx : end_token_idx + 1]
      extracted_text = (
        " "
        .join(token_slice)
        .replace(" - ", "-")
        .replace(" ( ", " (")
        .replace(" ) ", ") ")
      )

      datasaur_ground_truth.append({"role": role_label, "text": extracted_text.strip()})

  print(f"✅ Successfully loaded Row 0 directly from nested structures.")
  print(f'📝 Text Target: "{sentence_text}"\n')

except Exception as e:
  raise RuntimeError(f"❌ Failed loading data components: {str(e)}")

In [ ]:
# =====================================================================
# DIVISION 4: TWIN CHRONOLOGICAL TABLES EVALUATION MATRIX
# =====================================================================
try:
  # 1. Retrieve the structural JSON array from the local model
  llm_predictions = query_ollama_zero_shot(sentence=sentence_text, verb=target_action)

  # Helper tracking maps for evaluation processing
  actual_map = {arg["role"]: arg["text"] for arg in datasaur_ground_truth}
  predicted_map = {
    pred.get("role", "UNKNOWN"): pred.get("text", "") for pred in llm_predictions
  }

  # Helper sort algorithm: finds where a substring starts in the master text.
  # If the text is missing or hallucinated out-of-sentence, handles gracefully.
  def get_token_start_index(text_span: str) -> int:
    if text_span and text_span in sentence_text:
      return sentence_text.index(text_span)
    return 999

  # -----------------------------------------------------------------
  # TABLE 1: DATASAUR GROUND TRUTH PROFILE (Ordered Left-to-Right)
  # -----------------------------------------------------------------
  # Sorts your ground truth array by its actual text occurrence index
  ordered_ground_truth = sorted(
    datasaur_ground_truth, key=lambda x: get_token_start_index(x["text"])
  )

  print("\n" + "=" * 125)
  print(f"📋 TABLE 1: DATASAUR GROUND TRUTH BASELINE (Chronological Sentence Flow)")
  print("=" * 125)
  print(
    f"{'Actual SRL Role':<18} | {'Ground Truth Text Span (Expected Target)':<60} | Prediction Alignment Status"
  )
  print("-" * 125)

  for item in ordered_ground_truth:
    role = item["role"]
    actual_text = item["text"]
    pred_text = predicted_map.get(role, "")

    # Check if the LLM successfully captured the exact string for this specific role
    if actual_text.strip().lower() == pred_text.strip().lower():
      status = f"✅ EXACT MATCH (Predicted: '{role}')"
    elif pred_text:
      status = f"⚠️ LABEL/SPAN SHIFT (Predicted: '{pred_text}')"
    else:
      status = "❌ OMITTED / MISSED BY LLM"

    print(f"{role:<18} | {actual_text:<60} | {status}")
  print("=" * 125)

  # -----------------------------------------------------------------
  # TABLE 2: LLM OUTPUT PROFILE (Ordered Left-to-Right)
  # -----------------------------------------------------------------
  # Sorts the LLM's returned objects by where its extractions sit in the text
  ordered_llm_predictions = sorted(
    llm_predictions, key=lambda x: get_token_start_index(x.get("text", ""))
  )

  print("\n" + "=" * 125)
  print(f"🤖 TABLE 2: LOCAL OLLAMA INTERPRETATION PROFILE ({MODEL_NAME})")
  print("=" * 125)
  print(
    f"{'Predicted Role':<18} | {'LLM Extracted Verbatim Text Span':<60} | Ground Truth Validity"
  )
  print("-" * 125)

  for pred in ordered_llm_predictions:
    p_role = pred.get("role", "UNKNOWN")
    p_text = pred.get("text", "[No text extracted]")

    # Cross-reference the LLM's choice back against the valid handbook tags
    actual_text_for_this_role = actual_map.get(p_role, "")

    if p_role not in VALID_ROLES:
      validity = "❌ FABRICATED ROLE (Not in Dataset Codebook)"
    elif p_text.strip().lower() == actual_text_for_this_role.strip().lower():
      validity = "✅ VALID MATCH WITH GROUND TRUTH"
    else:
      validity = "⚠️ VARIANT SPAN BOUNDARY (Annotators carved this section differently)"

    print(f"{p_role:<18} | {p_text:<60} | {validity}")
  print("=" * 125)

except Exception as e:
  print(f"\n❌ Pipeline execution comparison crashed: {str(e)}")


In [ ]:
datasaur_ground_truth

## Results